# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedKroush/Flyrank-ML1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os
import getpass

# Get Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

# Build March 2026 page-level dataset
labeled = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {fact_daily}
        WHERE report_date >= '2026-03-01'
          AND report_date < '2026-04-01'
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 15 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last15,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 15 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev15,

            AVG(f.gsc_avg_position) AS avg_position_month,

            SUM(f.gsc_impressions) AS impressions_month,
            SUM(f.gsc_clicks) AS clicks_month

        FROM {fact_daily} f, bounds b

        WHERE f.report_date >= '2026-03-01'
          AND f.report_date < '2026-04-01'

        GROUP BY 1, 2

        HAVING imp_prev15 >= 50
    )

    SELECT *,
           (imp_last15 < 0.8 * imp_prev15)::INT AS is_declining
    FROM windowed
""").df()

print(f"Rows loaded: {len(labeled):,}")
print(f"Observed decline base rate: {labeled['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 94,559
Observed decline base rate: 0.373


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The review queue prioritizes pages using observed search-performance signals rather than automatically changing content. Pages with meaningful impression volume and weak average position receive a higher priority because they combine measurable visibility with room for review. Pages showing an observed decline are also prioritized, with reason codes explaining whether the queue was driven by decline, weak position, or both.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the editorial review queue

queue = labeled.copy()

queue["priority_score"] = (
    queue["is_declining"] * 2
    + (queue["impressions_month"] >= 250).astype(int)
    + (queue["avg_position_month"] > 20).astype(int)
)

def reason_code(row):
    reasons = []

    if row["is_declining"] == 1:
        reasons.append("OBSERVED_DECLINE")

    if row["impressions_month"] >= 250:
        reasons.append("MEANINGFUL_VISIBILITY")

    if row["avg_position_month"] > 20:
        reasons.append("WEAK_POSITION")

    if not reasons:
        reasons.append("MONITOR")

    return "+".join(reasons)

queue["reason_code"] = queue.apply(reason_code, axis=1)

def recommended_action(row):
    if row["is_declining"] == 1 and row["avg_position_month"] > 20:
        return "PRIORITY_REVIEW"
    elif row["is_declining"] == 1:
        return "REVIEW_DECLINE"
    elif row["avg_position_month"] > 20:
        return "POSITION_REVIEW"
    else:
        return "MONITOR"

queue["recommended_action"] = queue.apply(
    recommended_action,
    axis=1
)

queue = queue.sort_values(
    ["priority_score", "impressions_month"],
    ascending=[False, False]
)

print("=== TOP REVIEW QUEUE ===")

print(
    queue[
        [
            "content_hash_id",
            "priority_score",
            "reason_code",
            "recommended_action",
            "impressions_month",
            "clicks_month",
            "avg_position_month",
            "is_declining"
        ]
    ].head(20).to_string(index=False)
)

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

=== TOP REVIEW QUEUE ===
         content_hash_id  priority_score                                          reason_code recommended_action  impressions_month  clicks_month  avg_position_month  is_declining
content_36e53e9c707674fc               4 OBSERVED_DECLINE+MEANINGFUL_VISIBILITY+WEAK_POSITION    PRIORITY_REVIEW           194579.0         242.0           32.766674             1
content_3df3f32f3fd58dea               4 OBSERVED_DECLINE+MEANINGFUL_VISIBILITY+WEAK_POSITION    PRIORITY_REVIEW           140156.0         197.0           23.335465             1
content_bdf60c86117079be               4 OBSERVED_DECLINE+MEANINGFUL_VISIBILITY+WEAK_POSITION    PRIORITY_REVIEW           112429.0          12.0           30.769353             1
content_559cdd76da9306de               4 OBSERVED_DECLINE+MEANINGFUL_VISIBILITY+WEAK_POSITION    PRIORITY_REVIEW            97378.0           2.0           36.712074             1
content_9fff53e827550f9d               4 OBSERVED_DECLINE+MEANINGFUL_VISIBI

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The queue is intended for FlyRank editors or analysts who need to decide which content pages to review first. It is decision-support, not an automatic content optimizer or a prediction of Google's ranking system. The recommendations are based on observed relationships in the available dataset and may become less reliable when traffic patterns, clients, measurement coverage, or the underlying search environment changes.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summarize intended-use output

print("=== INTENDED USE ===")
print("Primary user: content editor / SEO analyst")
print("Purpose: prioritize pages for human review")
print("Output: priority score, reason code, recommended action")

print("\n=== LIMITS ===")
print("Not an automatic content editor")
print("Not a prediction of Google's algorithm")
print("Not causal evidence")
print("Requires human review before action")

=== INTENDED USE ===
Primary user: content editor / SEO analyst
Purpose: prioritize pages for human review
Output: priority score, reason code, recommended action

=== LIMITS ===
Not an automatic content editor
Not a prediction of Google's algorithm
Not causal evidence
Requires human review before action


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before acting on a recommendation, a human should check search intent, content quality, freshness, relevance, technical issues, seasonality, and whether the observed change is large enough to matter operationally. The system should never automatically rewrite, delete, merge, publish, or materially alter a page based only on the priority score. It should also never expose client-identifying information or private search data.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Human-review checklist

review_checks = [
    "Search intent",
    "Content quality",
    "Content freshness",
    "Topical relevance",
    "Technical issues",
    "Seasonality or unusual traffic patterns",
    "Magnitude of observed change"
]

no_go_actions = [
    "Automatic rewriting",
    "Automatic deletion",
    "Automatic merging",
    "Automatic publishing",
    "Automatic ranking claims"
]

print("=== HUMAN REVIEW CHECKLIST ===")
for item in review_checks:
    print(f"- {item}")

print("\n=== NO-GO AUTOMATION ===")
for item in no_go_actions:
    print(f"- {item}")

=== HUMAN REVIEW CHECKLIST ===
- Search intent
- Content quality
- Content freshness
- Topical relevance
- Technical issues
- Seasonality or unusual traffic patterns
- Magnitude of observed change

=== NO-GO AUTOMATION ===
- Automatic rewriting
- Automatic deletion
- Automatic merging
- Automatic publishing
- Automatic ranking claims


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The queue should be reviewed when the observed decline base rate changes materially, when the distribution of impressions or clicks shifts, or when the relationship between priority scores and observed outcomes weakens. A substantial change in the client mix or measurement coverage is also a reason to reassess the scoring rules and retrain or recalibrate the model before relying on it operationally.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Current monitoring reference values

print("=== MONITORING BASELINE ===")

print(
    f"Current observed decline rate: "
    f"{labeled['is_declining'].mean():.3f}"
)

print(
    f"Median monthly impressions: "
    f"{labeled['impressions_month'].median():.1f}"
)

print(
    f"Median monthly clicks: "
    f"{labeled['clicks_month'].median():.1f}"
)

print(
    f"Median average position: "
    f"{labeled['avg_position_month'].median():.1f}"
)

print("\nReassess when these distributions or the decline rate shift materially.")

=== MONITORING BASELINE ===
Current observed decline rate: 0.373
Median monthly impressions: 859.0
Median monthly clicks: 1.0
Median average position: 8.6

Reassess when these distributions or the decline rate shift materially.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked queue is exported for reuse in the deployed research paper. Pseudonymous identifiers are retained only where needed to distinguish rows internally; no client names, domains, private queries, credentials, or other client-identifying information are exported.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Export the ranked queue for the paper

os.makedirs("work/outputs", exist_ok=True)

export_cols = [
    "content_hash_id",
    "priority_score",
    "reason_code",
    "recommended_action",
    "impressions_month",
    "clicks_month",
    "avg_position_month",
    "is_declining"
]

queue[export_cols].head(100).to_csv(
    "work/outputs/content_review_queue.csv",
    index=False
)

print("=== EXPORTS ===")
print("Saved:")
print("work/outputs/content_review_queue.csv")

=== EXPORTS ===
Saved:
work/outputs/content_review_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.